In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("odins0n/ucf-crime-dataset")

print("Path to dataset files:", path)

100%|██████████| 11.0G/11.0G [01:33<00:00, 126MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/odins0n/ucf-crime-dataset/versions/1


In [3]:
import os
os.listdir(path)

['Test', 'Train']

In [6]:
tr=os.path.join(path,"Train")
te=os.path.join(path,"Test")
print(os.listdir(tr))
print(os.listdir(te))

['Vandalism', 'Abuse', 'NormalVideos', 'Assault', 'Shoplifting', 'Arrest', 'Shooting', 'Stealing', 'Fighting', 'Burglary', 'RoadAccidents', 'Robbery', 'Explosion', 'Arson']
['Vandalism', 'Abuse', 'NormalVideos', 'Assault', 'Shoplifting', 'Arrest', 'Shooting', 'Stealing', 'Fighting', 'Burglary', 'RoadAccidents', 'Robbery', 'Explosion', 'Arson']


In [7]:
classes = [
    'Abuse',
    'Arrest',
    'Arson',
    'Assault',
    'Burglary',
    'Explosion',
    'Fighting',
    'NormalVideos',
    'Robbery',
    'RoadAccidents',
    'Shooting',
    'Shoplifting',
    'Stealing',
    'Vandalism'
]

In [8]:
classes = sorted(os.listdir(tr))

label_map = {
    cls:i
    for i,cls in enumerate(classes)
}

print(label_map)

{'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}


In [9]:
train_data = []

for cls in classes:

    class_path = os.path.join(tr, cls)

    if not os.path.isdir(class_path):
        continue

    label = label_map[cls]

    for video in os.listdir(class_path):

        train_data.append(
            (
                os.path.join(class_path, video),
                label
            )
        )

print(len(train_data))

1266345


In [10]:
test_data = []

for cls in classes:

    class_path = os.path.join(te, cls)

    if not os.path.isdir(class_path):
        continue

    label = label_map[cls]

    for video in os.listdir(class_path):

        test_data.append(
            (
                os.path.join(class_path, video),
                label
            )
        )

print(len(test_data))

111308


In [12]:
import cv2
import numpy as np

In [13]:
IMG_SIZE = 128
SEQ_LEN = 20
def extract_frames(video_path):

    cap = cv2.VideoCapture(video_path)

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    step = max(
        total_frames // SEQ_LEN,
        1
    )

    frames = []

    for i in range(SEQ_LEN):

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            i * step
        )

        success, frame = cap.read()

        if not success:
            break

        frame = cv2.resize(
            frame,
            (IMG_SIZE, IMG_SIZE)
        )

        frame = frame / 255.0

        frames.append(frame)

    cap.release()

    while len(frames) < SEQ_LEN:

        frames.append(
            np.zeros(
                (
                    IMG_SIZE,
                    IMG_SIZE,
                    3
                )
            )
        )

    return np.array(frames)

In [14]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

x = GlobalAveragePooling2D()(
    base_model.output
)

cnn_model = Model(
    inputs=base_model.input,
    outputs=x
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [15]:
def extract_video_features(video_path):

    frames = extract_frames(video_path)

    features = cnn_model.predict(
        frames,
        verbose=0
    )

    return features

In [16]:
LIMIT = 1500

X_train = []
y_train = []

for video_path, label in train_data[:LIMIT]:

    try:

        features = extract_video_features(
            video_path
        )

        X_train.append(features)

        y_train.append(label)

    except:
        pass

In [17]:
X_train = np.array(X_train)

y_train = np.array(y_train)

print(X_train.shape)
print(y_train.shape)

(1500, 20, 1280)
(1500,)


In [18]:
from tensorflow.keras.utils import to_categorical

NUM_CLASSES = len(classes)

y_train = to_categorical(
    y_train,
    num_classes=NUM_CLASSES
)

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

model = Sequential()

model.add(
    LSTM(
        128,
        input_shape=(20,1280)
    )
)

model.add(
    Dropout(0.3)
)

model.add(
    Dense(
        64,
        activation='relu'
    )
)

model.add(
    Dense(
        NUM_CLASSES,
        activation='softmax'
    )
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │       721,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │           910 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 730,574 (2.79 MB)

 Trainable params: 730,574 (2.79 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9867 - loss: 0.0767 - val_accuracy: 1.0000 - val_loss: 1.2027e-04
Epoch 2/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 1.0000 - loss: 1.0587e-04 - val_accuracy: 1.0000 - val_loss: 2.1577e-05
Epoch 3/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 1.0000 - loss: 2.4911e-05 - val_accuracy: 1.0000 - val_loss: 7.9870e-06
Epoch 4/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 1.0000 - loss: 1.2387e-05 - val_accuracy: 1.0000 - val_loss: 4.0531e-06
Epoch 5/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 1.0000 - loss: 6.9034e-06 - val_accuracy: 1.0000 - val_loss: 2.1458e-06
Epoch 6/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 1.0000 - loss: 3.9615e-06 - val_accuracy: 1.0000 - val_loss: 1.5497e-06
Epoch 7/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 1.0000 - loss: 3.3047e-06 - val_accuracy: 1.0000 - val_loss: 9.5367e-07
Epoch 8/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accura

In [21]:
model.save(
    "crime_classifier.h5"
)

cnn_model.save(
    "cnn_feature_extractor.h5"
)

In [22]:
video_path = test_data[0][0]

feature = extract_video_features(
    video_path
)

feature = np.expand_dims(
    feature,
    axis=0
)

pred = model.predict(feature)

idx = np.argmax(pred)

print(classes[idx])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
Abuse


In [23]:
idx_to_class = {
    v:k
    for k,v in label_map.items()
}

print(idx_to_class)

{0: 'Abuse', 1: 'Arrest', 2: 'Arson', 3: 'Assault', 4: 'Burglary', 5: 'Explosion', 6: 'Fighting', 7: 'NormalVideos', 8: 'RoadAccidents', 9: 'Robbery', 10: 'Shooting', 11: 'Shoplifting', 12: 'Stealing', 13: 'Vandalism'}


In [24]:
prediction = idx_to_class[np.argmax(pred)]